# 03 — Agent memory patterns: sessions, events, scratch

**Audience:** developers building agent-style memory — sessions, events, and
working memory (scratch) — on the memtomem Python API.

memtomem ships with first-class primitives for agent-style memory:

- **Sessions** record the lifespan of an agent run. Each session can hold a
  sequence of events.
- **Events** are timestamped records of what the agent did (`search`,
  `decide`, `observe`, …).
- **Scratch (working memory)** is a key-value store that lives for the
  duration of a task, optionally scoped to a session and with a TTL.

In this notebook we walk through one agent-style turn end-to-end, using the
storage mixins directly. Everything runs against a temp directory and local
ONNX embeddings — no server required.

## Prerequisites

- **memtomem** with the ONNX extra — local embeddings, **no server**:
  ```bash
  # From PyPI
  uv pip install "memtomem[onnx]" jupyter ipykernel
  # Or from a source checkout
  uv pip install -e "packages/memtomem[onnx]" jupyter ipykernel
  ```

This notebook is mostly DB-only (sessions, events, scratch); only the final
time-based recall step indexes a note, which uses the local ONNX model.

In [1]:
# Confirm the ONNX embedding backend is importable before doing anything else.
# fastembed runs the model locally in-process, so there is no server to check.
try:
    import fastembed  # noqa: F401
except ModuleNotFoundError as e:
    raise RuntimeError(
        "fastembed is not installed. Install the ONNX extra and re-run this cell:\n"
        '  uv pip install "memtomem[onnx]" jupyter ipykernel'
    ) from e

print("ONNX embedding backend available.")

ONNX embedding backend available.


## Step 1 — Set up components

In [2]:
import json
import os
import tempfile
from pathlib import Path

from memtomem.config import Mem2MemConfig
from memtomem.server.component_factory import (
    Components,
    create_components,
    close_components,
)

# 1. Create an isolated temp directory so nothing lands in ~/.memtomem.
tmp = tempfile.TemporaryDirectory()
tmp_path = Path(tmp.name)
db_path = tmp_path / "memory.db"
notes_dir = tmp_path / "notes"
notes_dir.mkdir()

# 2. Override config via environment variables. Note the double underscore —
#    pydantic-settings uses "__" as the nesting separator for section fields.
#    The ONNX provider runs locally via fastembed (no server); "bge-small-en-v1.5"
#    is the short alias for BAAI/bge-small-en-v1.5 (384-dim, ~67 MB).
os.environ["MEMTOMEM_STORAGE__SQLITE_PATH"] = str(db_path)
os.environ["MEMTOMEM_INDEXING__MEMORY_DIRS"] = json.dumps([str(notes_dir)])
os.environ["MEMTOMEM_EMBEDDING__PROVIDER"] = "onnx"
os.environ["MEMTOMEM_EMBEDDING__MODEL"] = "bge-small-en-v1.5"
os.environ["MEMTOMEM_EMBEDDING__DIMENSION"] = "384"

# 3. Apply the same fields directly on the config object, and temporarily
#    disable load_config_overrides() so the real ~/.memtomem/config.json
#    cannot leak into this notebook. This mirrors the pattern used in
#    packages/memtomem/tests/conftest.py.
config = Mem2MemConfig()
config.storage.sqlite_path = db_path
config.indexing.memory_dirs = [notes_dir]

import memtomem.config as _cfg

_orig_loader = _cfg.load_config_overrides
_cfg.load_config_overrides = lambda c: None
try:
    comp: Components = await create_components(config)
finally:
    _cfg.load_config_overrides = _orig_loader

print(f"memtomem components ready. db={db_path}")

memtomem components ready. db=/tmp/tmpf9st9_2j/memory.db


## Step 2 — Start a session

Sessions need a UUID you supply yourself. The `agent_id` is a free-form
label for whichever agent is running, and `namespace` is a memtomem
namespace that lets you segment episodic memory by project.

In [3]:
import uuid
from datetime import datetime, timezone

session_id = str(uuid.uuid4())

await comp.storage.create_session(
    session_id=session_id,
    agent_id="demo-agent",
    namespace="research",
    metadata={"task": "investigate retrieval quality"},
)

print(f"session started: {session_id}")

session started: 4f44079c-ff9c-44a1-bd9c-9dfd8abb574e


## Step 3 — Log events as the agent works

Each call to `add_session_event()` appends a row to the session log. The
`event_type` is free-form; common values are `search`, `observe`, `decide`,
`tool_call`, `reflect`, `error`. The `chunk_ids` list lets you link an event
back to the retrieval result that justified it.

In [4]:
await comp.storage.add_session_event(
    session_id=session_id,
    event_type="search",
    content="hybrid retrieval tuning",
)

await comp.storage.add_session_event(
    session_id=session_id,
    event_type="observe",
    content="default rrf_k=60 works for short queries, suffers on long ones",
)

await comp.storage.add_session_event(
    session_id=session_id,
    event_type="decide",
    content="try rrf_k=30 + rerank=True for queries > 16 tokens",
)

print("three events logged")

three events logged


## Step 4 — Use scratch (working memory) for intermediate state

Scratch entries are meant for short-lived, task-scoped values — things you
would normally put on an agent's blackboard or in a dict passed through
node state. Scoping an entry to `session_id` lets memtomem clean it up when
the session ends.

In [5]:
await comp.storage.scratch_set(
    key="current_hypothesis",
    value="long queries benefit from lower rrf_k",
    session_id=session_id,
)

await comp.storage.scratch_set(
    key="pending_todo",
    value="re-run tuning on the 200-query eval set",
    session_id=session_id,
)

fetched = await comp.storage.scratch_get("current_hypothesis")
print("current_hypothesis:", fetched["value"] if fetched else "(missing)")

scratch_rows = await comp.storage.scratch_list(session_id=session_id)
print(f"scratch entries for this session: {len(scratch_rows)}")
for row in scratch_rows:
    print(f"  {row['key']:20s} -> {row['value']}")

current_hypothesis: long queries benefit from lower rrf_k
scratch entries for this session: 2
  current_hypothesis   -> long queries benefit from lower rrf_k
  pending_todo         -> re-run tuning on the 200-query eval set


## Step 5 — Replay the event log

At any time you can ask memtomem for the full event stream of a session.
This is how you would rebuild an agent's trajectory or show a human
reviewer what happened.

In [6]:
events = await comp.storage.get_session_events(session_id)

print(f"{len(events)} event(s) in session {session_id[:8]}…")
for e in events:
    print(f"  [{e['event_type']:8s}] {e['content']}")

3 event(s) in session 4f44079c…
  [search  ] hybrid retrieval tuning
  [observe ] default rrf_k=60 works for short queries, suffers on long ones
  [decide  ] try rrf_k=30 + rerank=True for queries > 16 tokens


## Step 6 — End the session

`end_session()` records a summary plus arbitrary metadata (we tuck the
event type counts in there for bookkeeping). After the session ends the
scratch entries that were scoped to it can be dropped via
`scratch_cleanup(session_id=...)`.

In [7]:
event_counts: dict[str, int] = {}
for e in events:
    event_counts[e["event_type"]] = event_counts.get(e["event_type"], 0) + 1

await comp.storage.end_session(
    session_id=session_id,
    summary="explored rrf_k tradeoff, planned eval re-run",
    metadata={"event_counts": event_counts},
)

dropped = await comp.storage.scratch_cleanup(session_id=session_id)
print(f"session closed, {dropped} scratch entries cleaned up")
print("event counts:", event_counts)

session closed, 2 scratch entries cleaned up
event counts: {'search': 1, 'observe': 1, 'decide': 1}


## Step 7 — Time-based recall

`storage.recall_chunks()` is the building block behind the `mem_recall`
MCP tool. It returns chunks created inside a time window, with optional
source and namespace filters — no query embedding required. We will index
one note, then recall everything that was added in the last minute.

In [8]:
# Drop a fresh note so we have something to recall.
recall_note = notes_dir / "finding.md"
recall_note.write_text(
    "# Retrieval finding\n\n"
    "Hybrid mode beats dense-only on short factual queries but loses on "
    "long narrative queries, probably because BM25 saturation dominates.\n",
    encoding="utf-8",
)
await comp.index_engine.index_file(recall_note)

from datetime import timedelta

since = datetime.now(timezone.utc) - timedelta(minutes=1)

recent = await comp.storage.recall_chunks(since=since, limit=10)
print(f"{len(recent)} chunk(s) in the last minute:")
for chunk in recent:
    name = Path(chunk.metadata.source_file).name
    head = " / ".join(chunk.metadata.heading_hierarchy) or "(no heading)"
    print(f"  {name:20s} {head}")

[06/27/26 12:56:07] INFO     Loading ONNX embedding model BAAI/bge-small-en-v1.5 (threads=4,             ]8;id=910299;file:///Users/you/memtomem/packages/memtomem/src/memtomem/embedding/onnx.py\onnx.py]8;;\:]8;id=192491;file:///Users/you/memtomem/packages/memtomem/src/memtomem/embedding/onnx.py#81\81]8;;\
                             cache_dir=/Users/you/.memtomem/cache/fastembed) …                                

1 chunk(s) in the last minute:
  finding.md           # Retrieval finding


## Mapping to the MCP tool surface

Most calls above have an MCP-tool equivalent. In `core` tool mode the session
and scratch tools live behind the `mem_do` meta-tool; time-based recall uses
the core `mem_recall` tool directly.

| Python call | MCP equivalent |
|-------------|----------------|
| `storage.create_session(...)` | `mem_do(action="session_start")` |
| `storage.end_session(...)` | `mem_do(action="session_end")` |
| `storage.scratch_set(...)` | `mem_do(action="scratch_set")` |
| `storage.scratch_get(...)` | `mem_do(action="scratch_get")` |
| `storage.recall_chunks(...)` | `mem_recall(...)` |

Per-event logging (`add_session_event`) and event replay (`get_session_events`)
are Python-API-only — there is no standalone MCP tool for them. An agent driving
memtomem over MCP records the session lifecycle with the tools above and gets
the event summary back from `mem_session_end`.

## Step 8 — Per-project agent teams (ADR-0028)

Running one agent *team* per project against a single memtomem server? Keep
each project's memory separate with two namespace conventions — no special
schema required:

- **Private per-agent memory** — encode the project in the `agent_id`:
  `projA-planner` derives the namespace `agent-runtime:projA-planner`.
  Project B's team uses `projB-*`. The literal ids differ, so the teams
  never collide, and `agent-runtime:` stays hidden from a default
  `mem_search`.
- **Team-shared memory** — use a per-project shared bucket
  `shared:<project>` (e.g. `shared:projA`) instead of the single global
  `shared`, so two projects' teams don't pool into the same bucket.

A read composes both legs with a comma-joined namespace filter — exactly
what `mem_agent_search(agent_id="projA-planner", shared_namespace="shared:projA")`
builds internally. The orthogonal *scope* axis (`user` / `project_shared` /
`project_local`, ADR-0011) is about file residency, not team identity, and
is left untouched here.

In [ ]:
# Seed memory for two project teams. Each team has a private per-agent
# namespace (agent-runtime:<project>-<role>) and a per-project shared
# bucket (shared:<project>). One markdown file per namespace.
team_notes = {
    "agent-runtime:projA-planner": "Planner: ship the payments API behind a feature flag.",
    "shared:projA": "Team decision: use HTTP 409 (not 422) for a duplicate charge id.",
    "agent-runtime:projB-researcher": "Researcher: compare vector-only vs hybrid recall.",
    "shared:projB": "Team decision: default to hybrid retrieval with reranking.",
}

for ns, text in team_notes.items():
    slug = ns.replace(":", "__")  # unique, filesystem-safe file name per namespace
    note = notes_dir / f"{slug}.md"
    note.write_text(f"# {ns}\n\n{text}\n", encoding="utf-8")
    await comp.index_engine.index_file(note, namespace=ns)

print(f"seeded {len(team_notes)} notes across two project teams")

In [ ]:
# Project A's planner reads its private namespace + project A's shared
# bucket. This comma-joined filter is exactly what mem_agent_search builds
# for agent_id="projA-planner", shared_namespace="shared:projA".
projA_filter = "agent-runtime:projA-planner,shared:projA"
results, _ = await comp.search_pipeline.search(
    query="team decision", top_k=10, namespace=projA_filter
)

print(f"Project A planner sees ({projA_filter}):")
for r in results:
    ns = r.chunk.metadata.namespace
    last_line = r.chunk.content.strip().splitlines()[-1]
    print(f"  [{ns}] {last_line}")

# Project B lives in different namespaces, so it is filtered out at the
# storage layer — it cannot leak into project A's scope regardless of
# how relevant the query is to it.
leaked = [r for r in results if "projB" in (r.chunk.metadata.namespace or "")]
assert not leaked, "project B memory must not appear in project A's scope"
print("\n✓ No project B memory leaked into project A's scope.")

### Over MCP

The same seed-and-read pattern maps onto the multi-agent MCP tools:

| Python here | MCP equivalent |
|-------------|----------------|
| `index_file(..., namespace="agent-runtime:projA-planner")` | `mem_session_start(agent_id="projA-planner")` then `mem_add(...)` |
| `index_file(..., namespace="shared:projA")` | `mem_add(namespace="shared:projA")` / `mem_agent_share(target="shared:projA")` |
| `search_pipeline.search(namespace="agent-runtime:projA-planner,shared:projA")` | `mem_agent_search(agent_id="projA-planner", shared_namespace="shared:projA")` |

`shared_namespace=` keeps the shared merge inside the project; omit it and
`include_shared=True` falls back to the single global `shared` bucket. See
**ADR-0028** for the full rationale and the rejected alternatives (namespace
nesting, the scope axis).

## Cleanup

In [9]:
# Release connections and remove the temp directory.
await close_components(comp)
tmp.cleanup()

# Clean up the environment variables we set.
for key in (
    "MEMTOMEM_STORAGE__SQLITE_PATH",
    "MEMTOMEM_INDEXING__MEMORY_DIRS",
    "MEMTOMEM_EMBEDDING__PROVIDER",
    "MEMTOMEM_EMBEDDING__MODEL",
    "MEMTOMEM_EMBEDDING__DIMENSION",
):
    os.environ.pop(key, None)

print("clean.")

clean.
